In [1]:
# -*- coding: utf-8 -*-

# =========================
# Install (Colab 用)
# =========================
!pip install pybullet

# =========================
# Imports
# =========================
import os
import math
import numpy as np
import pybullet as p
import pybullet_data

# =========================
# Config
# =========================
EPISODE_ID = 123
SAVE_DIR = "dataset"

BLOCK_MASS = 1.0
BLOCK_FRICTION = 0.5

TIME_STEPS = 800
DELTA_TIME = 1.0 / 240.0

# action = [push_speed, push_force]
PUSH_SPEED = 0.0008
PUSH_FORCE = 800

# =========================
# PyBullet Init
# =========================
p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.8)
p.setTimeStep(DELTA_TIME)

# =========================
# Environment
# =========================
p.loadURDF("plane.urdf")

# --- Wide table ---
table_col = p.createCollisionShape(
    p.GEOM_BOX,
    halfExtents=[0.5, 0.5, 0.02]
)
table_vis = p.createVisualShape(
    p.GEOM_BOX,
    halfExtents=[0.5, 0.5, 0.02],
    rgbaColor=[0.6, 0.4, 0.3, 1]
)
tableId = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=table_col,
    baseVisualShapeIndex=table_vis,
    basePosition=[0.6, 0, 0.62]
)

# =========================
# Robot (KUKA iiwa)
# =========================
robotId = p.loadURDF(
    "kuka_iiwa/model.urdf",
    basePosition=[0, 0, 0],
    useFixedBase=True
)

num_joints = p.getNumJoints(robotId)
ee_link = 6

initial_q = [0, 0.3, 0, -1.2, 0, 1.0, 0.5]
for i in range(num_joints):
    p.resetJointState(robotId, i, initial_q[i])

# =========================
# Block (white)
# =========================
block_half_extents = [0.05, 0.05, 0.02]

block_col = p.createCollisionShape(
    p.GEOM_BOX,
    halfExtents=block_half_extents
)
block_vis = p.createVisualShape(
    p.GEOM_BOX,
    halfExtents=block_half_extents,
    rgbaColor=[1, 1, 1, 1]
)

blockId = p.createMultiBody(
    baseMass=BLOCK_MASS,
    baseCollisionShapeIndex=block_col,
    baseVisualShapeIndex=block_vis,
    basePosition=[0.55, 0, 0.66]
)

p.changeDynamics(
    blockId,
    -1,
    lateralFriction=BLOCK_FRICTION,
    angularDamping=0.95,
    linearDamping=0.05
)

# =========================
# Camera (64x64 RGB)
# =========================
view_matrix = p.computeViewMatrix(
    cameraEyePosition=[1.2, 0, 1.0],
    cameraTargetPosition=[0.6, 0, 0.65],
    cameraUpVector=[0, 0, 1]
)
proj_matrix = p.computeProjectionMatrixFOV(
    fov=60,
    aspect=1.0,
    nearVal=0.1,
    farVal=3.0
)

def capture_rgb():
    img = p.getCameraImage(
        64, 64,
        viewMatrix=view_matrix,
        projectionMatrix=proj_matrix
    )
    rgb = np.reshape(img[2], (64, 64, 4))[:, :, :3]
    return rgb.astype(np.uint8)

# =========================
# Log Buffers
# =========================
rgb_buf = []
q_buf = []
dq_buf = []
f_buf = []
action_buf = []
block_pose_buf = []

# =========================
# Simulation Loop
# =========================
x = 0.55
y = 0.0
z = 0.68

for t in range(TIME_STEPS):

    # --- target pose ---
    x += PUSH_SPEED
    target_pos = [x, y, z]
    target_ori = p.getQuaternionFromEuler([0, math.pi / 2, 0])

    joint_poses = p.calculateInverseKinematics(
        robotId,
        ee_link,
        target_pos,
        target_ori
    )

    for j in range(num_joints):
        p.setJointMotorControl2(
            robotId,
            j,
            p.POSITION_CONTROL,
            joint_poses[j],
            force=PUSH_FORCE
        )

    p.stepSimulation()

    # =================
    # Logging
    # =================
    rgb_buf.append(capture_rgb())

    q, dq, ff = [], [], []
    joint_states = p.getJointStates(robotId, range(num_joints))
    for js in joint_states:
        q.append(js[0])
        dq.append(js[1])
        ff.append(js[3])

    q_buf.append(q)
    dq_buf.append(dq)
    f_buf.append(ff)

    action_buf.append([PUSH_SPEED, PUSH_FORCE])

    pos, ori = p.getBasePositionAndOrientation(blockId)
    block_pose_buf.append(list(pos) + list(ori))

# =========================
# Save Episode (.npz)
# =========================
os.makedirs(SAVE_DIR, exist_ok=True)

filename = (
    f"episode_{EPISODE_ID:06d}"
    f"_m={BLOCK_MASS:.2f}"
    f"_mu={BLOCK_FRICTION:.2f}.npz"
)

path = os.path.join(SAVE_DIR, filename)

np.savez_compressed(
    path,
    rgb=np.array(rgb_buf),
    q=np.array(q_buf, dtype=np.float32),
    dq=np.array(dq_buf, dtype=np.float32),
    f=np.array(f_buf, dtype=np.float32),
    action=np.array(action_buf, dtype=np.float32),
    block_pose=np.array(block_pose_buf, dtype=np.float32),
    mass=float(BLOCK_MASS),
    friction=float(BLOCK_FRICTION)
)

print(f"✅ Saved episode: {path}")

p.disconnect()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873468 sha256=336e0dfa471b0ca5bd57bba9ab8e6b4f2b15c95b8c0623088ef6dc6de1cf8485
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet
✅ Saved episode: dataset/episode_000123_m=1.00_mu=0.50.npz


In [ ]:
import numpy as np

data = np.load("./content/dataset/episode_000123_m=1.0_mu=0.5.npz")

print(data.files)

FileNotFoundError: [Errno 2] No such file or directory: './content/dataset/episode_000123_m=1.0_mu=0.5.npz'